In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Data Preprocessing
new_data_path = "C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/test.csv"
df_new = pd.read_csv(new_data_path)

# Dropping unnecessary columns
df_new.drop(['id', 'classification'], axis=1, inplace=True)

# Renaming columns for user-friendliness
df_new.columns = ['age', 'blood_pressure', 'specific_gravity', 'albumin', 'sugar', 'red_blood_cells', 'pus_cell',
              'pus_cell_clumps', 'bacteria', 'blood_glucose_random', 'blood_urea', 'serum_creatinine', 'sodium',
              'potassium', 'haemoglobin', 'packed_cell_volume', 'white_blood_cell_count', 'red_blood_cell_count',
              'hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'appetite', 'peda_edema',
              'anemia']

# Replace incorrect values
df_new['diabetes_mellitus'].replace(to_replace={'\tno': 'no', '\tyes': 'yes', ' yes': 'yes'}, inplace=True)
df_new['coronary_artery_disease'].replace(to_replace='\tno', value='no', inplace=True)

# Convert columns to numerical type
df_new['packed_cell_volume'] = pd.to_numeric(df_new['packed_cell_volume'], errors='coerce')
df_new['white_blood_cell_count'] = pd.to_numeric(df_new['white_blood_cell_count'], errors='coerce')
df_new['red_blood_cell_count'] = pd.to_numeric(df_new['red_blood_cell_count'], errors='coerce')

# Define column mappings
column_mappings = {
    'red_blood_cells': {'normal': 0, 'abnormal': 1},
    'pus_cell': {'normal': 0, 'abnormal': 1},
    'pus_cell_clumps': {'notpresent': 0, 'present': 1},
    'bacteria': {'notpresent': 0, 'present': 1},
    'hypertension': {'no': 0, 'yes': 1},
    'diabetes_mellitus': {'no': 0, 'yes': 1},
    'coronary_artery_disease': {'no': 0, 'yes': 1},
    'appetite': {'good': 0, 'poor': 1},
    'peda_edema': {'no': 0, 'yes': 1},
    'anemia': {'no': 0, 'yes': 1}
}

# Apply custom mappings to the selected columns
df_new.replace(column_mappings, inplace=True)

# Data Normalization
biomarker_inputs = df_new.to_numpy()

# Define default values for each column
column_defaults = {
    'age': 51.483376,
    'blood_pressure': 76.469072,
    'specific_gravity': 1.017408,
    'albumin': 0,
    'sugar': 0,
    'red_blood_cells': 0,
    'pus_cell': 1,
    'pus_cell_clumps': 0,
    'bacteria': 0,
    'blood_glucose_random': 120,
    'blood_urea': 57.425722,
    'serum_creatinine': 0.450142,
    'sodium': 137.528754,
    'potassium': 4.627244,
    'haemoglobin': 12.526437,
    'packed_cell_volume': 45,
    'white_blood_cell_count': 7000,
    'red_blood_cell_count': 5,
    'hypertension': 0,
    'diabetes_mellitus': 0,
    'coronary_artery_disease': 0,
    'appetite': 0,
    'peda_edema': 0,
    'anemia': 0
}

# Replace null values with custom default values using an if-else condition
for col_idx, col_name in enumerate(column_defaults.keys()):
    default_value = column_defaults[col_name]
    biomarker_inputs[:, col_idx] = np.where(np.isnan(biomarker_inputs[:, col_idx]),
                                             default_value, biomarker_inputs[:, col_idx])

# Define model architecture
class KidneyCNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(KidneyCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * (input_size - 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Define model path and test data path
model_path = 'C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/Final/trained_model2.pt'
test_data_path = "C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/test.csv"

# Load the model and make predictions
def predict_ckd(test_data_path, model_path):
    # Load the test data
    df_new = pd.read_csv(test_data_path)

    # Data preprocessing (same as above)
    # ...

    # Load the model
    loaded_model = KidneyCNN(input_size=24, num_classes=2)
    loaded_model.load_state_dict(torch.load(model_path))
    loaded_model.eval()  # Set the model to evaluation mode

    # Convert test data to PyTorch tensor
    X_test_tensor = torch.tensor(biomarker_inputs, dtype=torch.float32)

    # Perform predictions
    with torch.no_grad():
        outputs = loaded_model(X_test_tensor)
        class_probs = torch.softmax(outputs, dim=1)  # Apply softmax to get probabilities
        predicted_class_index = torch.argmax(class_probs, dim=1).item()
        confidence_score = class_probs[0, predicted_class_index].item()

    # Get class mapping and print results
    class_mapping = {'ckd': 0, 'notckd': 1}
    predicted_class = list(class_mapping.keys())[list(class_mapping.values()).index(predicted_class_index)]

    return predicted_class, confidence_score

# Make predictions
predicted_class, confidence_score = predict_ckd(test_data_path, model_path)

# Print the results
print(f"Predicted class: {predicted_class}")
print(f"Confidence score: {confidence_score:.4f}")


Predicted class: ckd
Confidence score: 1.0000
